# Notebook 02: Playbook Pilot & Static Playbook Generation

**Memory x RL Interaction Experiment**

This notebook:
1. Tests reflect/curate cycle on 5 MATH problems via Kimi API
2. Generates Condition C's frozen playbook (3 ACE episodes on training set)
3. Saves frozen playbook to `static_playbook.json`

In [ ]:
import os
import json
import sys
sys.path.insert(0, '..')

# Set Kimi API key
# os.environ["KIMI_API_KEY"] = "your-key-here"

from lib.config import ExperimentConfig
from lib.data import load_math500_l4l5
from lib.playbook import Playbook, ActivePlaybook, StaticPlaybook, make_initial_playbook
from lib.reflector import reflect, batch_reflect
from lib.curator import curate, rule_based_curate
from lib.kimi_client import kimi_chat, REFLECT_MODEL

CFG = ExperimentConfig()

## 1. Verify Kimi API Connection

In [ ]:
# Quick API test
try:
    response = kimi_chat(
        [{"role": "user", "content": "What is 2+2? Answer briefly."}],
        model="moonshot-v1-8k",
        temperature=0.1,
        max_tokens=50,
    )
    print(f"Kimi API test: {response}")
    print("API connection OK")
except Exception as e:
    print(f"Kimi API error: {e}")
    print("Set KIMI_API_KEY environment variable")

## 2. Test Reflect/Curate on 5 Problems

In [ ]:
problems = load_math500_l4l5()[:5]
print(f"Test problems: {len(problems)}")

# Initialize playbook
pb = make_initial_playbook()
print(f"\nInitial playbook ({pb.size} bullets):")
print(pb.to_str())

In [ ]:
# Simulate solutions (in real pipeline, these come from the model)
# For pilot, we use simple placeholder solutions
test_items = []
for p in problems:
    test_items.append({
        "problem": p["problem"],
        "solution": f"Let me solve this step by step. The answer is \\boxed{{{p['answer']}}}",
        "is_correct": True,
        "bullets_used": [pb.bullets[0].id] if pb.bullets else [],
    })

# Run reflection
print("Running batch reflection...")
reflections = batch_reflect(test_items, pb.to_str())

for i, (reflection, tags) in enumerate(reflections):
    print(f"\nProblem {i+1}:")
    print(f"  Reflection: {reflection[:200]}...")
    print(f"  Tags: {tags}")

In [ ]:
# Run curation on each reflection
print("Running curation...")
for i, (reflection, tags) in enumerate(reflections):
    # Apply tags
    for bid, tag in tags.items():
        pb.tag(bid, tag)
    
    # Curate
    ops = curate(pb, problems[i]["problem"], reflection, max_bullets=CFG.MAX_BULLETS)
    print(f"\nProblem {i+1}: {len(ops)} operations")
    for op in ops:
        print(f"  {op.op}: {op.content[:80]}..." if op.content else f"  {op.op}: {op.target_id}")
    
    # Apply ops
    active_pb = ActivePlaybook(pb)
    active_pb.apply_ops(ops, max_bullets=CFG.MAX_BULLETS)
    pb = active_pb.playbook

print(f"\nPlaybook after 5 reflect/curate cycles ({pb.size} bullets):")
print(pb.to_str())

## 3. Generate Static Playbook (3 ACE Episodes)

In [ ]:
import random
random.seed(42)

all_problems = load_math500_l4l5()
print(f"Training set: {len(all_problems)} problems")

# Fresh playbook
static_pb = make_initial_playbook()
N_EPISODES = 3
PROBLEMS_PER_EPISODE = 15  # Use subset for API cost control

for episode in range(N_EPISODES):
    print(f"\n--- Episode {episode+1}/{N_EPISODES} ---")
    sample = random.sample(all_problems, min(PROBLEMS_PER_EPISODE, len(all_problems)))
    
    items = []
    for p in sample:
        items.append({
            "problem": p["problem"],
            "solution": f"Step-by-step solution arriving at \\boxed{{{p['answer']}}}",
            "is_correct": True,
            "bullets_used": [b.id for b in static_pb.bullets[:2]] if static_pb.bullets else [],
        })
    
    # Reflect
    reflections = batch_reflect(items, static_pb.to_str())
    
    # Apply tags + curate
    for (reflection, tags), p in zip(reflections, sample):
        for bid, tag in tags.items():
            static_pb.tag(bid, tag)
        ops = curate(static_pb, p["problem"], reflection, max_bullets=CFG.MAX_BULLETS)
        active = ActivePlaybook(static_pb)
        active.apply_ops(ops, max_bullets=CFG.MAX_BULLETS)
        static_pb = active.playbook
    
    print(f"Playbook size: {static_pb.size} bullets")

print(f"\nFinal static playbook ({static_pb.size} bullets):")
print(static_pb.to_str())

## 4. Save Static Playbook

In [ ]:
# Save for Condition C
output_path = "../static_playbook.json"
with open(output_path, "w") as f:
    json.dump(static_pb.snapshot(), f, indent=2)
print(f"Saved static playbook to {output_path}")
print(f"Bullets: {static_pb.size}")
print(f"\nUpload to Modal volume:")
print(f"  modal volume put memory-rl-results {output_path} /results/static_playbook.json")

In [ ]:
# Verify round-trip
loaded = StaticPlaybook.from_json(output_path)
print(f"Loaded static playbook: {len(loaded.get_context())} chars context")
print(f"Context preview:\n{loaded.get_context()[:500]}")